In [1]:
import os
import warnings
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from pytorch_lightning import (LightningDataModule, LightningModule, Trainer)
from pytorch_lightning.callbacks import EarlyStopping, ModelCheckpoint, Timer
from pytorch_lightning.loggers import WandbLogger
import wandb
import pandas as pd
import polars as pl
import numpy as np
from sklearn.metrics import r2_score
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset, DataLoader


class custom_args():
    def __init__(self):
        self.usegpu = True
        self.gpuid = 2
        self.seed = 42
        self.model = 'nn'
        self.use_wandb = False
        self.project = 'js-xs-nn-with-lags'
        self.dname = "./input_df/"
        self.loader_workers = 4
        self.bs = 8192
        self.lr = 1e-3
        self.weight_decay = 5e-4
        self.dropouts = [0.1, 0.1]
        self.n_hidden = [512, 512, 256]
        self.patience = 25
        self.max_epochs = 1000
        self.N_fold = 5


my_args = custom_args()

/home/jx1421/JS-Kaggle-JX/js-venv/lib64/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [7]:
valid = pl.scan_parquet("valid.parquet").collect().to_pandas()


In [10]:
def add_moving_averages(df, columns_to_transform, date_col='date_id', window=4, min_periods=1):
    for col in columns_to_transform:
        new_col_name = f"{col}_ma{window}"
        df[new_col_name] = (
            df.groupby(date_col)[col]
              .transform(lambda x: x.rolling(window=window, min_periods=min_periods).mean())
        )
    return df

valid = pl.scan_parquet("valid.parquet").collect().to_pandas()
valid = add_moving_averages(valid,[f"feature_{i:02d}" for i in range(79)], window=4)
valid.head()

,id,date_id,time_id,symbol_id,weight,feature_00,feature_01,feature_02,feature_03,feature_04,...,feature_69_ma4,feature_70_ma4,feature_71_ma4,feature_72_ma4,feature_73_ma4,feature_74_ma4,feature_75_ma4,feature_76_ma4,feature_77_ma4,feature_78_ma4
0,46045114,1670,0,0,3.405333,0.310848,1.131483,0.745982,0.776556,-0.798888,...,-0.321387,-0.737133,0.032996,-0.614959,NaN,NaN,0.811054,0.708597,-0.001946,0.004254
1,46045115,1670,0,1,2.619899,0.382185,0.458687,0.544353,0.092550,-0.911843,...,-0.170661,-0.654957,1.182334,-0.412229,NaN,NaN,0.835208,0.674913,-0.026510,-0.014172
2,46045116,1670,0,2,2.213198,-0.163449,0.740222,0.272134,-0.092383,-1.215421,...,-0.391011,-0.855707,0.694581,-0.473789,NaN,NaN,0.861053,0.919690,-0.015167,-0.003567
3,46045117,1670,0,3,1.603719,0.362074,0.242651,0.463201,0.282871,-1.410926,...,-0.380276,-0.875254,0.582183,-0.549475,NaN,NaN,1.054770,1.036527,0.041979,0.071481
4,46045118,1670,0,5,2.820571,-0.015899,1.502831,0.238775,0.330908,-0.765390,...,-0.334627,-0.894339,0.720156,-0.452454,NaN,NaN,0.911412,0.917130,0.006392,0.032494


In [9]:
def add_moving_averages_gpt(df, columns_to_transform, date_col='date_id', window=4, min_periods=1):
    # Create a dictionary to hold the new columns
    new_columns = {}
    for col in columns_to_transform:
        new_col_name = f"{col}_ma{window}"
        new_columns[new_col_name] = (
            df.groupby(date_col)[col]
              .transform(lambda x: x.rolling(window=window, min_periods=min_periods).mean())
        )
    new_columns_df = pd.DataFrame(new_columns)
    df = pd.concat([df, new_columns_df], axis=1)
    return df

valid = pl.scan_parquet("valid.parquet").collect().to_pandas()
valid = add_moving_averages_gpt(valid, [f"feature_{i:02d}" for i in range(79)], window=4)
valid.head()


,id,date_id,time_id,symbol_id,weight,feature_00,feature_01,feature_02,feature_03,feature_04,...,feature_69_ma4,feature_70_ma4,feature_71_ma4,feature_72_ma4,feature_73_ma4,feature_74_ma4,feature_75_ma4,feature_76_ma4,feature_77_ma4,feature_78_ma4
0,46045114,1670,0,0,3.405333,0.310848,1.131483,0.745982,0.776556,-0.798888,...,-0.321387,-0.737133,0.032996,-0.614959,NaN,NaN,0.811054,0.708597,-0.001946,0.004254
1,46045115,1670,0,1,2.619899,0.382185,0.458687,0.544353,0.092550,-0.911843,...,-0.170661,-0.654957,1.182334,-0.412229,NaN,NaN,0.835208,0.674913,-0.026510,-0.014172
2,46045116,1670,0,2,2.213198,-0.163449,0.740222,0.272134,-0.092383,-1.215421,...,-0.391011,-0.855707,0.694581,-0.473789,NaN,NaN,0.861053,0.919690,-0.015167,-0.003567
3,46045117,1670,0,3,1.603719,0.362074,0.242651,0.463201,0.282871,-1.410926,...,-0.380276,-0.875254,0.582183,-0.549475,NaN,NaN,1.054770,1.036527,0.041979,0.071481
4,46045118,1670,0,5,2.820571,-0.015899,1.502831,0.238775,0.330908,-0.765390,...,-0.334627,-0.894339,0.720156,-0.452454,NaN,NaN,0.911412,0.917130,0.006392,0.032494


In [3]:
window_size = 4
features_for_ma =  ["feature_10", "feature_42", "feature_04", "feature_06"]
features_for_var = ['feature_20', 'feature_21', 'feature_10','feature_15']
feature_names = [f"feature_{i:02d}" for i in range(79)] + \
                [f"responder_{idx}_lag_1" for idx in range(9)] + \
                [f"{col}_ma{window_size}" for col in features_for_ma] + \
                [f"{col}_var{window_size}" for col in features_for_var]
label_name = 'responder_6'
weight_name = 'weight'
class CustomDataset(Dataset):
    def __init__(self, df, accelerator):
        self.features = torch.FloatTensor(df[feature_names].values).to(accelerator)
        self.labels = torch.FloatTensor(df[label_name].values).to(accelerator)
        self.weights = torch.FloatTensor(df[weight_name].values).to(accelerator)
    
    def __len__(self):
        return len(self.labels)
    
    def __getitem__(self, idx):
        x = self.features[idx]
        y = self.labels[idx]
        w = self.weights[idx]
        return x, y, w


class DataModule(LightningDataModule):
    def __init__(self, train_df, batch_size, valid_df=None, accelerator='cpu'):
        super().__init__()
        self.df = train_df
        self.batch_size = batch_size
        self.dates = self.df['date_id'].unique()
        self.accelerator = accelerator
        self.train_dataset = None
        self.valid_df = None
        if valid_df is not None:
            self.valid_df = valid_df
        self.val_dataset = None

    def setup(self, fold=0, N_fold=5, stage=None):
        # Split dataset
        selected_dates = [date for ii, date in enumerate(self.dates) if ii % N_fold != fold]
        df_train = self.df.loc[self.df['date_id'].isin(selected_dates)]
        self.train_dataset = CustomDataset(df_train, self.accelerator)
        if self.valid_df is not None:
            df_valid = self.valid_df
            self.val_dataset = CustomDataset(df_valid, self.accelerator)

    def train_dataloader(self, n_workers=0):
        return DataLoader(self.train_dataset, batch_size=self.batch_size, shuffle=True, num_workers=n_workers)

    def val_dataloader(self, n_workers=0):
        return DataLoader(self.val_dataset, batch_size=self.batch_size, shuffle=False, num_workers=n_workers)
    

# Custom R2 metric for validation
def r2_val(y_true, y_pred, sample_weight):
    r2 = 1 - np.average((y_pred - y_true) ** 2, weights=sample_weight) / (np.average((y_true) ** 2, weights=sample_weight) + 1e-38)
    return r2


class NN(LightningModule):
    def __init__(self, input_dim, hidden_dims, dropouts, lr, weight_decay):
        super().__init__()
        self.save_hyperparameters()
        layers = []
        in_dim = input_dim
        for i, hidden_dim in enumerate(hidden_dims):
            layers.append(nn.BatchNorm1d(in_dim))
            if i > 0:
                layers.append(nn.SiLU())
            if i < len(dropouts):
                layers.append(nn.Dropout(dropouts[i]))
            layers.append(nn.Linear(in_dim, hidden_dim))
            # layers.append(nn.ReLU())
            in_dim = hidden_dim
        layers.append(nn.Linear(in_dim, 1)) 
        layers.append(nn.Tanh())
        self.model = nn.Sequential(*layers)
        self.lr = lr
        self.weight_decay = weight_decay
        self.validation_step_outputs = []

    def forward(self, x):
        return 5 * self.model(x).squeeze(-1)  

    def training_step(self, batch):
        x, y, w = batch
        y_hat = self(x)
        loss = F.mse_loss(y_hat, y, reduction='none') * w  #
        loss = loss.mean()
        self.log('train_loss', loss, on_step=False, on_epoch=True, batch_size=x.size(0))
        return loss

    def validation_step(self, batch):
        x, y, w = batch
        y_hat = self(x)
        loss = F.mse_loss(y_hat, y, reduction='none') * w
        loss = loss.mean()
        self.log('val_loss', loss, on_step=False, on_epoch=True, batch_size=x.size(0))
        self.validation_step_outputs.append((y_hat, y, w))
        return loss

    def on_validation_epoch_end(self):
        """Calculate validation WRMSE at the end of the epoch."""
        y = torch.cat([x[1] for x in self.validation_step_outputs]).cpu().numpy()
        if self.trainer.sanity_checking:
            prob = torch.cat([x[0] for x in self.validation_step_outputs]).cpu().numpy()
        else:
            prob = torch.cat([x[0] for x in self.validation_step_outputs]).cpu().numpy()
            weights = torch.cat([x[2] for x in self.validation_step_outputs]).cpu().numpy()
            # r2_val
            val_r_square = r2_val(y, prob, weights)
            self.log("val_r_square", val_r_square, prog_bar=True, on_step=False, on_epoch=True)
        self.validation_step_outputs.clear()

    def configure_optimizers(self):
        optimizer = torch.optim.Adam(self.parameters(), lr=self.lr, weight_decay=self.weight_decay)
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5,
                                                               verbose=True)
        return {
            'optimizer': optimizer,
            'lr_scheduler': {
                'scheduler': scheduler,
                'monitor': 'val_loss',
            }
        }

    def on_train_epoch_end(self):
        if self.trainer.sanity_checking:
            return
        epoch = self.trainer.current_epoch
        metrics = {k: v.item() if isinstance(v, torch.Tensor) else v for k, v in self.trainer.logged_metrics.items()}
        formatted_metrics = {k: f"{v:.5f}" for k, v in metrics.items()}
        print(f"Epoch {epoch}: {formatted_metrics}")

In [6]:
def add_moving_averages(df, columns_to_transform, date_col='date_id', window=4, min_periods=1):
    for col in columns_to_transform:
        new_col_name = f"{col}_ma{window}"
        df[new_col_name] = (
            df.groupby(date_col)[col]
              .transform(lambda x: x.rolling(window=window, min_periods=min_periods).mean())
        )
    return df

# now add function for sample var in sliding window
def add_sample_var(df, columns_to_transform, date_col='date_id', window=4, min_periods=1):
    for col in columns_to_transform:
        new_col_name = f"{col}_var{window}"
        df[new_col_name] = (
            df.groupby(date_col)[col]
              .transform(lambda x: x.rolling(window=window, min_periods=min_periods).var())
        )
    return df

In [7]:
df = pl.scan_parquet('train.parquet').collect().to_pandas()
valid = pl.scan_parquet("valid.parquet").collect().to_pandas()
print(valid.shape)
valid = add_moving_averages(valid, features_for_ma, window=window_size)
valid = add_sample_var(valid, features_for_var, window=window_size)
print(valid.shape)

(1082224, 104)
(1082224, 112)


In [25]:
valid = pl.scan_parquet("valid.parquet").collect().to_pandas()
print(valid.shape)
valid.columns[:50]

(1082224, 104)


Index(['id', 'date_id', 'time_id', 'symbol_id', 'weight', 'feature_00',
       'feature_01', 'feature_02', 'feature_03', 'feature_04', 'feature_05',
       'feature_06', 'feature_07', 'feature_08', 'feature_09', 'feature_10',
       'feature_11', 'feature_12', 'feature_13', 'feature_14', 'feature_15',
       'feature_16', 'feature_17', 'feature_18', 'feature_19', 'feature_20',
       'feature_21', 'feature_22', 'feature_23', 'feature_24', 'feature_25',
       'feature_26', 'feature_27', 'feature_28', 'feature_29', 'feature_30',
       'feature_31', 'feature_32', 'feature_33', 'feature_34', 'feature_35',
       'feature_36', 'feature_37', 'feature_38', 'feature_39', 'feature_40',
       'feature_41', 'feature_42', 'feature_43', 'feature_44'],
      dtype='object')

In [26]:
valid.columns[50:]

Index(['feature_45', 'feature_46', 'feature_47', 'feature_48', 'feature_49',
       'feature_50', 'feature_51', 'feature_52', 'feature_53', 'feature_54',
       'feature_55', 'feature_56', 'feature_57', 'feature_58', 'feature_59',
       'feature_60', 'feature_61', 'feature_62', 'feature_63', 'feature_64',
       'feature_65', 'feature_66', 'feature_67', 'feature_68', 'feature_69',
       'feature_70', 'feature_71', 'feature_72', 'feature_73', 'feature_74',
       'feature_75', 'feature_76', 'feature_77', 'feature_78', 'responder_0',
       'responder_1', 'responder_2', 'responder_3', 'responder_4',
       'responder_5', 'responder_6', 'responder_7', 'responder_8',
       'partition_id', 'label', 'responder_0_lag_1', 'responder_1_lag_1',
       'responder_2_lag_1', 'responder_3_lag_1', 'responder_4_lag_1',
       'responder_5_lag_1', 'responder_6_lag_1', 'responder_7_lag_1',
       'responder_8_lag_1'],
      dtype='object')

In [8]:
X_valid = valid[ feature_names ]
y_valid = valid[ label_name ]
w_valid = valid[ "weight" ]
X_valid.shape, y_valid.shape, w_valid.shape

((1082224, 96), (1082224,), (1082224,))

In [19]:
# Create multiple folds if you have an ensemble of NNs
N_folds = 4
models = []
model_path = '/home/jx1421/JS-Kaggle-JX/jx-stuff/lightning_logs'
for fold in range(N_folds):
    checkpoint_path = f"{model_path}/version_{fold+5}/checkpoints/MA-models/nn_{fold}.model.ckpt"
    print(checkpoint_path)
    # Load NN model, map to CUDA if available
    model = NN.load_from_checkpoint(
        checkpoint_path,
        map_location=lambda storage, loc: storage.cuda(0)
    )
    models.append(model.to("cuda:0"))

/home/jx1421/JS-Kaggle-JX/jx-stuff/lightning_logs/version_5/checkpoints/MA-models/nn_0.model.ckpt
/home/jx1421/JS-Kaggle-JX/jx-stuff/lightning_logs/version_6/checkpoints/MA-models/nn_1.model.ckpt
/home/jx1421/JS-Kaggle-JX/jx-stuff/lightning_logs/version_7/checkpoints/MA-models/nn_2.model.ckpt
/home/jx1421/JS-Kaggle-JX/jx-stuff/lightning_logs/version_8/checkpoints/MA-models/nn_3.model.ckpt


In [20]:
test_input = X_valid.fillna(method = 'ffill').fillna(0)
test_input = torch.FloatTensor(test_input.values).to("cuda:0")
print(test_input.shape)
preds = np.zeros((X_valid.shape[0],))
with torch.no_grad():
    for nn_model in models:
        nn_model.eval()
        pred_i = nn_model(test_input).cpu().numpy()
        print(r2_score(y_valid, pred_i, sample_weight=w_valid))
        preds += pred_i / len(models)
    print(f"predict> preds.shape =", preds.shape)


/tmp/ipykernel_395235/1398910070.py:1: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  test_input = X_valid.fillna(method = 'ffill').fillna(0)


torch.Size([1082224, 96])
0.006475269794464111
0.007812738418579102
0.007271528244018555
0.007943391799926758
predict> preds.shape = (1082224,)


In [21]:
y_pred_valid = preds
r2_score(y_valid, y_pred_valid, sample_weight=w_valid)

0.009150793627856535